Use before everything

In [6]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

192.168.88.1:50051


Line follow Advanced

In [ ]:
def line_follow(mult=4, speed=200):
    """Follow the detected line by turning proportionally to the line offset."""
    # Read line-tracking information from the robot.
    offset, line_type, x, y = got.get_single_track_total_info()

    # ADDED: Check if no line is detected
    if line_type == 0:
        print("Status: No line detected!")
        # Stop the motors so it doesn't drive off blindly
        got.mecanum_stop()
    else:
        # Convert the line offset into a turning speed.
        rotation_speed = int(offset * mult)
        # Move forward while rotating to stay aligned with the line.
        got.mecanum_move_xyz(x_speed=0, y_speed=speed, z_speed=rotation_speed)

    # Return the detection info so the main program can make decisions.
    return line_type, x, y





from IPython.display import clear_output

try:
    while True:
        # Set to detect a black line (line_type=0 in recognition settings)
        got.set_track_recognition_line(line_type=0)
        
        # Run the following logic
        line_type, x, y = line_follow(mult=0.25, speed=35)

        # 0 = no line detected
        if line_type == 0:
            clear_output(wait=True)
            print("Finished: No line detected. Stopping robot.")
            break
            
        # Optional: Print coordinates while it IS working
        clear_output(wait=True)
        print(f"Following line... X: {x}, Y: {y}")

    got.mecanum_stop()
    print("Done")

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Program Interrupted")

Line follow basic

In [ ]:
import time

def AP_centralization_approaching(distance=0.15, gap=20, fwd_spd=10, strafe_spd=10):
    """
    Drives toward an AprilTag with safety checks to prevent IndexErrors.
    """
    print("Approaching target...")
    
    while True:
        AP_info = got.get_apriltag_total_info()
        
        # ERROR FIX 1: Check if the list is empty before accessing index [0]
        if not AP_info or len(AP_info) == 0:
            print("Tag lost! Searching...")
            got.mecanum_stop()
            time.sleep(0.1) 
            continue # Skip to the next loop iteration to try and find it again

        # Safely extract data now that we know it exists
        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]

        # ERROR FIX 2: Combined movement for smoother pathing
        current_strafe = 0
        if AP_x < 320 - gap:
            current_strafe = -strafe_spd
        elif AP_x > 320 + gap:
            current_strafe = strafe_spd

        if AP_distance > distance:
            # Move forward and strafe at the same time
            got.mecanum_move_xyz(current_strafe, fwd_spd, 0)
        else:
            # We reached the goal
            got.mecanum_stop()
            print("Target reached. Stabilizing...")
            time.sleep(1.0) # ERROR FIX 3: Let inertia settle before picking up
            break

def pick_up():
    """
    Executes pickup sequence with validation logic.
    """
    AP_info = got.get_apriltag_total_info()
    
    # Validation: Ensure tag is still there before moving arm
    if not AP_info:
        print("Error: Tag disappeared right before pickup!")
        return

    AP_x = AP_info[0][1]
    AP_distance = AP_info[0][6]

    # 1. Prepare Arm
    # Using a safe mid-point for Joint 2 (30) so it doesn't hit the floor
    got.mechanical_joint_control(0, 30, 30, 1000)
    got.mechanical_clamp_release()
    time.sleep(1.5)

    # 2. Calculate Pose
    # Added 'max/min' clipping to keep values in safe mechanical ranges
    joint1 = int(max(min((AP_x - 320) * -0.1, 90), -90))
    joint3 = int(max(min(AP_distance * 100 - 80, 90), -90))

    # 3. Reach and Grab
    print(f"Executing Grab -> J1: {joint1}, J3: {joint3}")
    got.mechanical_joint_control(joint1, 0, joint3, 800)
    time.sleep(1.2)
    
    got.mechanical_clamp_close()
    time.sleep(1.5)

    # 4. Retract to safe carry position
    got.mechanical_joint_control(0, 30, 30, 1000)
    print("Pickup complete.")

# Main Execution
try:
    AP_centralization_approaching()
    pick_up()
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    got.mecanum_stop()


try:
    while True:
        # Poll the camera for any visible AprilTags.
        tag = got.get_apriltag_total_info()

        if tag:
            # A tag is visible — approach it and pick up the object.
            # Lower speeds (fwd_spd=5, strafe_spd=7) for more precise final approach.
            AP_centralization_approaching(distance= 1, gap=20, fwd_spd=5, strafe_spd=7)
            pick_up()

            print("Picked up!")
            break  # Exit after one successful pick-and-place

except KeyboardInterrupt:
    # Safety stop if the cell is interrupted manually.
    got.mecanum_stop()
    print("Done")


Approaching target...


KeyboardInterrupt: 

Line follow and pick up

In [ ]:
#later

Line follow with text dection

In [ ]:
#later